In [1]:
import requests
import pandas as pd
import re
from html import unescape

In [2]:
# 네이버 OpenAPI 인증 정보
CLIENT_ID = "EDZj4GtOZnrUuOC5quUq"
CLIENT_SECRET = "EV09c2U784"

# 네이버 뉴스 검색 OpenAPI 엔드포인트
url = "https://openapi.naver.com/v1/search/news.json"

headers = {
    "X-Naver-Client-Id": CLIENT_ID,
    "X-Naver-Client-Secret": CLIENT_SECRET
}

# 검색 조건 (display는 고정)
params = {
    "query": "인공지능",
    "display": 5,
    "start": 1
}

In [3]:
def clean_html(text):
    if text is None:
        return ""
    text = unescape(text)
    return re.sub(r"<.*?>", "", text)

In [6]:
# 문제 1) 뉴스 데이터를 누적 저장할 리스트를 생성하시오.
all_items = []

# 문제 2) start 값을 1, 6, 11로 변경하며 총 3번 API를 호출하시오.
for start in [1,6,11]:
    params["start"] = start

    # 문제 3) GET 요청을 보내시오.
    response = requests.get(url,headers=headers,params=params)

    # 문제 4) 상태 코드가 200이 아니면 에러를 발생시키시오.
    if response.status_code != 200:
        raise RuntimeError("API 호출 실패")

    # 문제 5) JSON 응답에서 items를 추출하시오.
    data = response.json()
    items = data['items']

    # 문제 6) 추출한 items를 all_items에 누적하시오.
    all_items.extend(items)

print("총 수집된 뉴스 개수:", len(all_items))

총 수집된 뉴스 개수: 15


In [7]:
# 문제 7) all_items에서 필요한 필드만 추출하여 rows 리스트를 완성하시오.
rows = []
for it in all_items:
    title = clean_html(it.get("title"))
    desc = clean_html(it.get("description"))

    rows.append({
        "title": title,
        "description": desc,
        "link": it.get("link"),
        "pubDate": it.get("pubDate"),
        # 문제 8) 제목의 글자 수를 계산하여 컬럼으로 추가하시오.
        "title_length": len(title)
    })

In [8]:
# 문제 9) rows를 DataFrame으로 변환하시오.
df = pd.DataFrame(rows)

print(df.head())
print("최종 데이터 개수:", len(df))

                                           title  \
0                   우원엠앤이-빔스온탑, 실무형 AI+BIM 협력 시동   
1  수은, 여신 88.6%가 무담보…"AI 기반 여신감리 조기경보모형 도입해 건...   
2     우리넷, 5G IoT와 광전송 기술 결합… '초연결·고보안' 두 토끼 잡는다   
3                      전기연-창원시 ‘개방형 제2캠퍼스’ 조성 협력   
4                          한미반도체 부회장 등극 김민현 미션은?   

                                         description  \
0  이번협약을 통해 양사는 인공지능(AI)과 BIM을 기반으로 한 설계 혁신과 산업 설...   
1  사진=한국수출입은행 한국수출입은행은 인공지능(AI) 기반 여신감리 조기경보모형 도입...   
2  광통신(광케이블 광섬유 등) 테마는 14일 인공지능(AI) 데이터센터 확충에 따른 ...   
3  제2캠퍼스는 인공지능(AI)과 전력반도체 등 국가 전략기술 분야에서 연구 협업을 비...   
4  곽 회장의 체질 개선 경영에서 한 발 더 나아가, 인공지능 반도체 시장의 급변하는 ...   

                                                link  \
0  https://www.dnews.co.kr/uhtml/view.jsp?idxno=2...   
1  https://daily.hankooki.com/news/articleView.ht...   
2  https://www.pinpointnews.co.kr/news/articleVie...   
3  https://www.knnews.co.kr/news/articleView.php?...   
4  http://www.globalepic.co.kr/view.php?ud=202604...   

             